In [1]:
# Data Ingestion & Cleaning
import pandas as pd
import numpy as np
import glob
import os
import re

## Data Ingestion & Cleaning
<b> Main Functions: </b>
- merge_product_files(data_folder) → Reads multiple CSVs, merges them, removes duplicates.
- clean_products(df) → Cleans raw product info, extracts unit price, standardizes base unit price, and computes discounts.

In [2]:
def merge_product_files(data_folder='data'):
    """
    Merges all CSV files in the specified folder into a single DataFrame
    
    Args:
        data_folder (str): Path to folder containing CSV files
        
    Returns:
        pd.DataFrame: Combined product data with source file tracking
    """
    # Find all CSV files in the folder
    all_files = glob.glob(os.path.join(data_folder, "*.csv"))
    
    # Read and concatenate files
    dfs = []
    for file in all_files:
        df = pd.read_csv(file)
        df['source_file'] = os.path.basename(file)  # Track origin
        dfs.append(df)
    
    # Combine with duplicate handling
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Remove exact duplicates (same data from multiple files)
    # combined_df.drop_duplicates(
    #     subset=['product_code'],  # Assuming this is your unique ID
    #     keep='first',
    #     inplace=True
    # )
    
    return combined_df

In [3]:
import pandas as pd
import numpy as np
import re

def clean_products(df):
    """
    Clean and standardise Coles product data for Smart Cart modelling.

    BUSINESS RULES IMPLEMENTED:
    ---------------------------------------------------------
    1. If unit_price is empty → treat as '1ea'
    2. If item_price invalid (NaN, text, <=0):
         - replace with best_price if valid
         - otherwise set both item_price and best_price to NaN
    3. Add binary flags:
         - on_special = 1 if special_text is not empty
         - on_promotion = 1 if promo_text is not empty
    4. Standardise extract_date → YYYY-MM-DD
    5. Standardise unit_of_measure:
         - valid units kept (g, kg, ml, l, ea)
         - partial units auto-corrected ('1k' → '1kg', '10m' → '10ml')
         - invalid units → '1ea'
    6. unit_price_value recalculated for invalid units using item_price
    7. base_unit_price standardised per 100g/ml for model comparability
    """

    df = df.copy()

    # ---------------------------------------------------------
    # STEP 1 — Remove duplicates
    # ---------------------------------------------------------
    df.drop_duplicates(subset=["product_code"], inplace=True)

    # ---------------------------------------------------------
    # STEP 2 — Fix item_price (RULE 2)
    # ---------------------------------------------------------
    df["item_price"] = pd.to_numeric(df["item_price"], errors="coerce")
    df["best_price"] = pd.to_numeric(df["best_price"], errors="coerce")

    # If item_price invalid → replace with best_price
    invalid_item = df["item_price"].isna() | (df["item_price"] <= 0)
    df.loc[invalid_item, "item_price"] = df.loc[invalid_item, "best_price"]

    # If item_price STILL invalid → both become NaN
    still_invalid = df["item_price"].isna() | (df["item_price"] <= 0)
    df.loc[still_invalid, ["item_price", "best_price"]] = np.nan

    # ---------------------------------------------------------
    # STEP 3 — Add on_special & on_promotion flags
    # ---------------------------------------------------------
    df["on_special"] = df["special_text"].fillna("").str.strip().ne("").astype(int)
    df["on_promotion"] = df["promo_text"].fillna("").str.strip().ne("").astype(int)

    # ---------------------------------------------------------
    # STEP 4 — Standardise dates
    # ---------------------------------------------------------
    if "extract_date" in df.columns:
        df["extract_date"] = pd.to_datetime(df["extract_date"], errors="coerce").dt.date

    # ---------------------------------------------------------
    # STEP 5 — Extract price + unit from unit_price column
    # ---------------------------------------------------------
    def extract_price_unit(text):
        if pd.isna(text):
            return (np.nan, np.nan)

        text = str(text).lower()
            # Normalize "/" to "per"   →   "$4/100g" → "$4 per 100g"
        text = text.replace("/", " per ")
        
        price_match = re.search(r"\$([\d\.]+)", text)
        unit_match = re.search(r"per\s*([a-zA-Z0-9]+)", text)

        price = float(price_match.group(1)) if price_match else np.nan
        unit = unit_match.group(1) if unit_match else np.nan
        return (price, unit)

    price_unit_list = df["unit_price"].apply(extract_price_unit).tolist()
    df[["unit_price_value", "unit_of_measure"]] = pd.DataFrame(price_unit_list, index=df.index)

    #    If unit_price_value is NaN → fallback to item_price
    # ---------------------------------------------------------
    df["unit_price_value"] = df["unit_price_value"].fillna(df["item_price"])

    # Optional: if still NaN, fall back to best_price
    df["unit_price_value"] = df["unit_price_value"].fillna(df["best_price"])

    #  If item_price is NaN → fallback to unit_price_value

    df["item_price"] = df["item_price"].fillna(df["unit_price_value"])    
    df["best_price"] = df["best_price"].fillna(df["unit_price_value"])

    df['discount_percentage'] = np.where( df['item_price'] > df['best_price'], (df['item_price'] - df['best_price']) / df['item_price'], 0 )

    # ---------------------------------------------------------
    # STEP 6 — Robust unit cleaning (RULE 1 + invalid units)
    # ---------------------------------------------------------
    def clean_unit(u):
        """
        Accepts raw unit text, returns valid forms of:
        '100g', '1kg', '100ml', '1l', '1ea'
        Converts invalid units → '1ea'
        Fixes partial units: '1k'→'1kg', '10m'→'10ml', '1kgm'→'1kg'
        """

        if pd.isna(u):
            return "1ea"

        u = str(u).lower().strip()

        # Partial kilograms: "1k" → "1kg"
        if re.fullmatch(r"\d+k", u):
            return u + "g"

        # Partial millilitres: "10m" → "10ml"
        if re.fullmatch(r"\d+m", u):
            return u + "l"

        # Fix "1kgm"
        if u.endswith("kgm"):
            return u.replace("kgm", "kg")

        # Valid patterns: number + unit suffix
        if re.fullmatch(r"\d+(g|kg|ml|l|ea)", u):
            return u

        # Anything else → invalid → treat as 1 each
        return "1ea"

    df["unit_of_measure"] = df["unit_of_measure"].apply(clean_unit)

    # ---------------------------------------------------------
    # STEP 7 — Fix unit_price_value when invalid units → item_price
    # ---------------------------------------------------------
    # df["unit_price_value"] = np.where(
    #     df["unit_of_measure"] == "1ea",
    #     df["item_price"],
    #     df["unit_price_value"]
    # )

    # ---------------------------------------------------------
    # STEP 8 — Standardised base_unit_price (per 100g/100ml)
    # ---------------------------------------------------------
    def base_price(row):
        unit = row["unit_of_measure"]
        price = row["unit_price_value"]

        if pd.isna(price):
            return np.nan

        # Grams
        if unit.endswith("g") and not unit.endswith("kg"):
            num = float(unit.replace("g", ""))
            return price / (num / 100)

        # Kilograms
        if unit.endswith("kg"):
            num = float(unit.replace("kg", ""))
            return price / (num * 10)  # because per 100g

        # Millilitres
        if unit.endswith("ml"):
            num = float(unit.replace("ml", ""))
            return price / (num / 100)

        # Litres
        if unit.endswith("l"):
            num = float(unit.replace("l", ""))
            return price / (num * 10)  # per 100ml

        # Each
        if unit.endswith("ea"):
            return price

        return price  # fallback

    df["base_unit_price"] = df.apply(base_price, axis=1)

    # ---------------------------------------------------------
    # FINAL RETURN
    # ---------------------------------------------------------
    return df


In [6]:
folder_path = 'data/scrapped_data'
products_df = merge_product_files(folder_path)   

C:\Users\rayed\AppData\Local\Temp\ipykernel_21276\148598269.py:17: DtypeWarning: Columns (2,4,5,8,11,12,13,14,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


In [5]:
# Ensure product_code is text to avoid scientific notation issues
products_df['product_code'] = products_df['product_code'].astype(str)

# Remove duplicate rows where product_code AND item_name are identical
products_df = products_df.drop_duplicates(
    subset=['product_code', 'item_name'],
    keep='first'   # keep the first occurrence
)



In [7]:
# Ensure product_code is treated as text
products_df['product_code'] = products_df['product_code'].astype(str)

# Find duplicated product codes
duplicate_codes = (
    products_df['product_code']
    .value_counts()
    .loc[lambda x: x > 1]
    .index
)

# Take the first code
first_code = duplicate_codes[5]

# Filter rows for that product_code
first_code_rows = products_df.loc[
    products_df['product_code'] == first_code,
    ['store', 'item_name', 'category']
]

# Show unique combinations only
first_code_unique = first_code_rows.drop_duplicates()

first_code_unique


,store,item_name,category
440,Coles,Favva Beans Sea Salt & Vinegar 6 Pack,SNACKS
18670,Coles,Fava Beans Sea Salt & Vinegar 6 Pack,SNACKS
37554,Coles,Happy Snack Company Favva Beans Sea Salt & Vin...,Pantry
37930,Coles,Happy Snack Company Favva Beans Sea Salt & Vin...,Back to School
37974,Coles,Happy Snack Company Favva Beans Sea Salt & Vin...,Dietary & World Foods
483293,NaN,Favva Beans Sea Salt & Vinegar 6 Pack,SNACKS


In [8]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN

c:\Users\rayed\anaconda3\envs\smartcart\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
products = products_df.copy()

# Make sure product_code is string
products['product_code'] = products['product_code'].astype(str)

# Clean item_name (remove nulls)
products = products[products['item_name'].notna()].reset_index(drop=True)

In [10]:
def clean_text(txt):
    txt = str(txt).lower()
    txt = txt.replace("(", " ").replace(")", " ").replace("/", " ")
    txt = txt.replace("-", " ").replace(",", " ").replace("ml", " ml").replace("kg", " kg")
    return " ".join(txt.split())

products['clean_name'] = products['item_name'].apply(clean_text)

In [19]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# 1. Pick a SMALL test subset
# -----------------------------

# Make sure product_code is string so grouping works properly
products_df['product_code'] = products_df['product_code'].astype(str)

# Find product codes that have more than one distinct name
codes_with_name_variants = (
    products_df.groupby('product_code')['item_name']
    .nunique()
    .loc[lambda x: x > 1]     # only codes with >1 different name
    .index
)

# Filter rows for only those codes
variant_rows = products_df[
    products_df['product_code'].isin(codes_with_name_variants)
][['product_code', 'item_name', 'category', 'store']].drop_duplicates()

# 👉 IMPORTANT: Limit to a small sample so it runs fast (change 30 → 50, 100 later)
test_df = variant_rows.head(100).reset_index(drop=True)

print("Testing on", len(test_df), "rows")
display(test_df.head(50))



Testing on 100 rows


,product_code,item_name,category,store
0,3455448.0,Pub Style Extra Crispy Fries,FROZEN VEGETABLES,Coles
1,7483558.0,Snack Crackers Mix 2 Multipack 12 Pack,BISCUITS & COOKIES,Coles
2,3706060.0,Snack Mix Variety Multipack 20 Pack,SNACKS,Coles
3,5376229.0,Chips BBQ,SNACKS,Coles
4,6696382.0,LCMs Split Stix Yoghurty 5 Pack,NUTRITIONAL SNACKS,Coles
5,6611459.0,Nutri-Grain Original Bars 5 Pack,NUTRITIONAL SNACKS,Coles
6,4975192.0,Chewy Choc Chip Muesli Bars School Lunchbox Sn...,NUTRITIONAL SNACKS,Coles
7,255646.0,Cup A Soup Creamy Chicken With Lots Of Noodles...,SOUP,Coles
8,332383.0,Crumpet Rounds Original,BAKERY SNACKS,Coles
9,2787974.0,Kitchen Asian Style Salad Kit,VALUE ADDED FP,Coles


In [20]:
# -----------------------------
# 2. Encode names with SBERT
# -----------------------------

model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode only this small list of names
name_embeddings = model.encode(test_df['item_name'].tolist())

In [21]:
# -----------------------------
# 3. Cluster similar names
# -----------------------------
# We cluster in embedding space using cosine distance
clusterer = DBSCAN(
    eps=0.25,          # how strict similarity is (lower = stricter)
    min_samples=2,
    metric='cosine'
)

labels = clusterer.fit_predict(name_embeddings)
test_df['cluster_id'] = labels

In [23]:

# -----------------------------
# 4. Show results
# -----------------------------

# Sort by cluster to see which names got grouped
test_df_sorted = test_df.sort_values(['cluster_id', 'product_code', 'item_name'])
display(test_df_sorted)
test_df_sorted.to_csv('test_df_sorted.csv')

,product_code,item_name,category,store,cluster_id
83,117256.0,Vita-Weat Original Crispbread Crackers,BISCUITS & COOKIES,Coles,-1
62,1224524.0,I'M Free From Anzac Biscuits,BISCUITS & COOKIES,Coles,-1
63,1224535.0,I'M Free From Stem Ginger Cookies,BISCUITS & COOKIES,Coles,-1
25,1777410.0,Cherry & Almond Bakewells Tarts 6 pack,BAKERY PACKAGED CAKE,Coles,-1
98,2366309.0,Apple Pineapple and Banana Tropical Smash,FRUIT-SHELF STABLE,Coles,-1
...,...,...,...,...,...
76,8221386.0,Nut Bars Peanut & Almond with Milk Choc Multip...,NUTRITIONAL SNACKS,Coles,12
56,1170130.0,Liquid Breakfast Vanilla Ice 12 x 250ml,CEREAL,Coles,13
55,8951356.0,Liquid Breakfast Choc Ice 12 x 250ml,CEREAL,Coles,13
59,3512064.0,Girl-Teen Regular Tampons,SANITARY PROTECTION,Coles,14


In [ ]:
# Optional: look only at rows that belong to real clusters (exclude noise = -1)
clustered_only = test_df_sorted[test_df_sorted['cluster_id'] != -1]
print("\nOnly clustered rows (ignoring noise):")
display(clustered_only)


In [ ]:
duplicates = products_df.groupby('product_code').filter(lambda x: x['item_name'].nunique() > 1)
duplicates[['product_code','item_name','store']].sort_values('product_code').head(30)

,product_code,item_name,store
470179,76.0,Up & Go Liquid Breakfast Strawberry 3 Pack,IGA
482184,76.0,Up & Go Liquid Breakfast Strawberry 3 Pack,IGA
448759,76.0,Up & Go Liquid Breakfast Strawberry 3 Pack,IGA
459889,76.0,Up & Go Liquid Breakfast Strawberry 3 Pack,IGA
395011,76.0,Up & Go Liquid Breakfast Strawberry,IGA
454376,76.0,Up & Go Liquid Breakfast Strawberry 3 Pack,IGA
476734,76.0,Up & Go Liquid Breakfast Strawberry 3 Pack,IGA
394185,563.0,Rexona Women Antiperspirant Aerosol Hypoallerg...,IGA
372916,563.0,Rexona Women Antiperspirant Aerosol Hypoallerg...,IGA
447929,563.0,Rexona Women Antiperspirant Aerosol Hypoallerg...,IGA


In [57]:
duplicates = products_df.groupby('product_code').filter(lambda x: x['item_name'].nunique() > 1)
duplicates[['product_code','item_name','store']].sort_values('product_code').head(30)


,product_code,item_name,store


In [42]:
products_df = clean_products(products_df)

In [43]:
products_df.columns

Index(['_id', 'product_code', 'category', 'item_name', 'best_price',
       'item_price', 'unit_price', 'special_text', 'promo_text', 'link',
       'extract_date', 'best_unit_price', 'price_was', 'timestamp',
       'category_slug', 'store', 'image', 'categories[0]', 'source_file',
       'on_special', 'on_promotion', 'unit_price_value', 'unit_of_measure',
       'discount_percentage', 'base_unit_price'],
      dtype='object')

In [44]:
products_df.isna().sum()

_id                        0
product_code               1
category                7037
item_name                 46
best_price              3418
item_price              3418
unit_price              4838
special_text           40493
promo_text             51513
link                       0
extract_date            4645
best_unit_price        36080
price_was              51199
timestamp              55329
category_slug          56275
store                      0
image                  50186
categories[0]          57148
source_file                0
on_special                 0
on_promotion               0
unit_price_value        3418
unit_of_measure            0
discount_percentage        0
base_unit_price         3418
dtype: int64

In [45]:
cols = ['product_code', 'category', 'item_name','store','extract_date','unit_price_value', 'unit_of_measure','base_unit_price', 'discount_percentage', 'on_special', 'on_promotion']
products_df_cleaned = products_df [cols] 


In [46]:
#products_df[products_df['unit_price'] == '$0.73 per 1ea']
products_df_cleaned[np.isnan(products_df_cleaned['unit_price_value'])]

,product_code,category,item_name,store,extract_date,unit_price_value,unit_of_measure,base_unit_price,discount_percentage,on_special,on_promotion
319858,8.025830e+05,Deli & Chilled Meals,Absolutely Wrapped Absolutely Wrapped Meat Lov...,Woolies,2024-08-06,NaN,1ea,NaN,0.0,0,0
325173,8.846180e+05,Pantry,Nature's Charm Oat Milk Sweetened Condensed Mi...,Woolies,2024-08-06,NaN,1ea,NaN,0.0,1,0
331701,9.962160e+05,Drinks,Great Northern Brewing Co Zero Alcohol Bottles...,Woolies,2024-08-06,NaN,1ea,NaN,0.0,1,0
331720,1.829850e+05,Drinks,Wolf Blass Zero Shiraz 750ml,Woolies,2024-08-06,NaN,1ea,NaN,0.0,0,0
331784,9.795090e+05,Drinks,Carlton Zero Non Alcoholic 375ml X 4 Pack,Woolies,2024-08-06,NaN,1ea,NaN,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
399668,9.319630e+05,Other,I CHOOSE RE-USE IGA Re-usable Bag,IGA,NaT,NaN,1ea,NaN,0.0,0,0
399669,9.319710e+05,Other,I CHOOSE RE-USE IGA Re-usable Bag,IGA,NaT,NaN,1ea,NaN,0.0,0,0
399670,1.800000e+11,Other,I CHOOSE RE-USE IGA Re-usable Red Bag,IGA,NaT,NaN,1ea,NaN,0.0,0,0
399671,1.800000e+11,Other,I Choose Re-Use Jute Bag,IGA,NaT,NaN,1ea,NaN,0.0,0,0


In [47]:
mask_blank_name = products_df_cleaned['item_name'].isna() | (products_df_cleaned['item_name'].str.strip() == "") 
#| products_df_remove_na['category'].isna() | (products_df_remove_na['category'].str.strip() == "")
products_df_cleaned = products_df_cleaned[~mask_blank_name]

In [48]:
# check null values
products_df_cleaned.isna().sum()

product_code              0
category               6991
item_name                 0
store                     0
extract_date           4645
unit_price_value       3417
unit_of_measure           0
base_unit_price        3417
discount_percentage       0
on_special                0
on_promotion              0
dtype: int64

In [49]:
products_df_cleaned = products_df_cleaned.dropna(subset=["unit_price_value", "extract_date"])
products_df_cleaned.isna().sum()

product_code              0
category               6989
item_name                 0
store                     0
extract_date              0
unit_price_value          0
unit_of_measure           0
base_unit_price           0
discount_percentage       0
on_special                0
on_promotion              0
dtype: int64

In [ ]:
products_df_cleaned.to_csv('data/products.csv')

In [50]:
repeated_codes = products_df_cleaned['product_code'].value_counts()
repeated_codes

product_code
3455448.0    1
7483558.0    1
3706060.0    1
7246880.0    1
5404110.0    1
            ..
776541.0     1
380073.0     1
380023.0     1
380031.0     1
186741.0     1
Name: count, Length: 49436, dtype: int64

In [ ]:
repeated_codes = repeated_codes[repeated_codes > 1].index.tolist()

In [52]:
repeated_codes

[]

In [28]:
repeated_codes = products_df_cleaned['product_code'].value_counts()
repeated_codes = repeated_codes[repeated_codes > 1].index.tolist()
repeated_products = products_df_cleaned[products_df_cleaned['product_code'].isin(repeated_codes)]
repeated_products[['product_code', 'item_name', 'category', 'store']] \
    .sort_values(['product_code', 'item_name'])


,product_code,item_name,category,store


In [38]:
products_df_cleaned.groupby(['product_code', 'store']).size().reset_index(name='count').head(20)

,product_code,store,count
0,25.0,Woolies,1
1,86.0,Woolies,1
2,128.0,IGA,1
3,168.0,Woolies,1
4,287.0,Woolies,1
5,326.0,Woolies,1
6,344.0,Woolies,1
7,563.0,IGA,1
8,579.0,Woolies,1
9,684.0,Woolies,1


In [ ]:
sample = df[df['product_code'] == df['product_code'].value_counts().index[0]]
sample[['product_code', 'item_name', 'extract_date', 'store', 'item_price']]
